# KND_04 - Step 3: Separate learning, practice and final testing

## Why are we doing this?

We need to check whether our model works on data it did not learn from.

- **Training data:** the model learns from these records.
- **Validation data:** we use these records to compare models and choose settings.
- **Test data:** we use these records for the final evaluation after our choices are fixed.

Think of training as studying, validation as a practice paper, and testing as the final exam. We should not use the final exam answers to choose how to study.

We will learn from older arrivals and evaluate on later arrivals. We will also keep each animal in only one split, although it can have multiple separate visits within that split.

## What you need

Finish Notebook 2 first. Keep its **step2_results** folder, including `labelled_records.csv` and `target_notes.json`, beside this notebook. In Colab, run Notebook 2 in the same session, or upload both of those files. Only pandas is needed in addition to Python's built-in libraries.

Run all cells in order. This notebook saves new files in **step3_results**, beside step2_results. It does not change the original CSV or Step 2 files. It does not fill missing ages, encode categories or train models.

Source: [County of Sonoma, Animal Shelter Intake and Outcome](https://data.sonomacounty.ca.gov/Government/Animal-Shelter-Intake-and-Outcome/924a-vesw), California, USA. We continue using our fixed September 2026 extract.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd

candidates = [
    Path("step2_results"), Path("outputs") / "step2_results",
    Path("/content/step2_results"), Path("."), Path("/content")
]
STEP2_DIR = next((p.resolve() for p in candidates
                  if (p / "labelled_records.csv").exists()
                  and (p / "target_notes.json").exists()), None)
if STEP2_DIR is None:
    raise FileNotFoundError("Run Notebook 2 first, or upload labelled_records.csv AND target_notes.json.")

SOURCE_PATH = STEP2_DIR / "labelled_records.csv"
source_hash = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()
notes = json.loads((STEP2_DIR / "target_notes.json").read_text(encoding="utf-8"))
df = pd.read_csv(SOURCE_PATH, dtype={"id": "string", "impound_number": "string"})

assert notes["source_sha256"] == "fd465b0dc5178bd13d0b9e70959320137f9526e635fe982c5ffc2c115faf61e0"
assert notes["observation_date"] == "2026-09-14"
assert len(df) == notes["retained_rows"] == 9670, "Check your Step 2 version."
assert df["source_row_number"].is_unique
assert df["id"].notna().all() and df["id"].str.strip().ne("").all()
assert df["long_stay_30"].isin([0, 1]).all()
assert df["exclusion_reason"].fillna("").eq("").all()

df["intake_date"] = pd.to_datetime(df["intake_date"], errors="raise").dt.normalize()
df["outcome_date"] = pd.to_datetime(df["outcome_date"], errors="raise").dt.normalize()
df = df.sort_values(["intake_date", "source_row_number"]).reset_index(drop=True)
OBSERVATION_DATE = pd.Timestamp(notes["observation_date"])
assert df["intake_date"].notna().all()
assert df["intake_date"].between("2023-01-01", "2026-08-14").all()
print("Loaded", len(df), "Step 2 records from", STEP2_DIR)

Loaded 9670 Step 2 records from C:\Users\ishin\Documents\Codex\2026-09-15\we-x20\outputs\step2_results


## 1. Choose the dates before modelling

Our choices are based on calendar periods, not model scores:

- **Training:** 1 January 2023 to 30 May 2025.
- **Gap:** 31 May to 30 June 2025.
- **Validation:** 1 July to 30 November 2025.
- **Gap:** 1 to 31 December 2025.
- **Testing:** 1 January to 14 August 2026.

This gives training more than two years of records, validation several later months, and testing the most recent year in our extract. The groups do not need to have exact 70/15/15 percentages.

**Why leave gaps?** An animal arriving just before validation begins might not yet have a known 30-day answer. A full 31-day gap gives earlier arrivals enough time to reach that answer. We also check the label dates below.

Keep these dates fixed while comparing models. Do not move them to get better scores.

In [2]:
VALIDATION_START = pd.Timestamp("2025-07-01")
TEST_START = pd.Timestamp("2026-01-01")
GAP_DAYS = 31
TRAIN_END_EXCLUSIVE = VALIDATION_START - pd.Timedelta(days=GAP_DAYS)
VALIDATION_END_EXCLUSIVE = TEST_START - pd.Timedelta(days=GAP_DAYS)

def assign_periods(intake_dates):
    period = pd.Series("gap", index=intake_dates.index, dtype="string")
    period.loc[intake_dates < TRAIN_END_EXCLUSIVE] = "train"
    period.loc[(intake_dates >= VALIDATION_START) &
               (intake_dates < VALIDATION_END_EXCLUSIVE)] = "validation"
    period.loc[intake_dates >= TEST_START] = "test"
    return period

df["candidate_split"] = assign_periods(df["intake_date"])
df["split_exclusion_reason"] = ""
df.loc[df["candidate_split"].eq("gap"), "split_exclusion_reason"] = "31-day gap before validation or testing"
print("Records before checking animal overlap:")
print(df["candidate_split"].value_counts().to_string())

Records before checking animal overlap:
candidate_split
train         6278
test          1672
validation    1201
gap            519


## 2. Check when the answer could have been known

- **Short stay:** the answer becomes known when the animal leaves. For a 12-day stay, that is the outcome date.
- **Long stay:** with our whole-day target, the answer is known once 31 days have passed. We do not need to wait for a later outcome to know the stay already exceeded 30 days.

For example, if an animal arrives on 1 January and leaves in March, its long-stay answer is already known on 1 February.

This calculates the earliest date implied by the recorded events. Our extract does not contain a full history of when staff entered or changed each record. We cannot prove that the information was actually entered on that day. This limitation, and the ongoing-stay assumption from Step 2, still apply.

In [3]:
def label_known_dates(records):
    known = records["outcome_date"].copy()
    long_stay = records["long_stay_30"].eq(1)
    known.loc[long_stay] = records.loc[long_stay, "intake_date"] + pd.Timedelta(days=31)
    return known

# Check that the labels still agree with the recorded dates.
completed = df["outcome_date"].notna()
duration = (df["outcome_date"] - df["intake_date"]).dt.days
assert (duration.loc[completed] >= 0).all()
assert (df.loc[completed, "outcome_date"] <= OBSERVATION_DATE).all()
assert df.loc[completed, "long_stay_30"].eq((duration.loc[completed] > 30).astype(int)).all()
assert df.loc[~completed, "long_stay_30"].eq(1).all()
assert df.loc[~completed, "outcome_type"].isna().all()

df["label_known_date"] = label_known_dates(df)
assert df["label_known_date"].notna().all()
assert (df["label_known_date"] <= OBSERVATION_DATE).all()

# These checks should remove zero extra rows because the gaps already allow 31 days.
late_train = df["candidate_split"].eq("train") & (df["label_known_date"] >= VALIDATION_START)
late_validation = df["candidate_split"].eq("validation") & (df["label_known_date"] >= TEST_START)
df.loc[late_train | late_validation, "split_exclusion_reason"] = "Label not known before the next period"
print("Extra rows set aside for label timing:", int((late_train | late_validation).sum()))

Extra rows set aside for label timing: 0


## 3. Keep the same animal out of different splits

Suppose the same dog arrived in 2024 and again in 2026. If its old visit is used for training and its new visit for testing, the model has already seen that animal.

Our main evaluation will measure performance on animals absent from the learning data:

1. Keep test-period visits.
2. Set aside validation visits belonging to test animals.
3. Set aside training visits belonging to validation or test animals.

We use **animal ID** for this check. The admission number identifies a visit and would not catch different visits by the same animal.

We remove earlier records; we do not move them into a later period. Multiple visits within a single split may remain because they are separate prediction events.

This is a deliberately stricter evaluation design. It uses later animal IDs to organise the retrospective evaluation, without using their outcomes to select model settings. It changes the population being measured and does not measure performance on returning animals already seen during training.

In [4]:
available = df["split_exclusion_reason"].eq("")
test_ids = set(df.loc[available & df["candidate_split"].eq("test"), "id"])
validation_ids = set(df.loc[available & df["candidate_split"].eq("validation"), "id"])

validation_overlap = available & df["candidate_split"].eq("validation") & df["id"].isin(test_ids)
training_overlap = available & df["candidate_split"].eq("train") & df["id"].isin(validation_ids | test_ids)

df.loc[validation_overlap, "split_exclusion_reason"] = "Validation animal also appears in test period"
df.loc[training_overlap, "split_exclusion_reason"] = "Training animal also appears in validation or test period"

df["final_split"] = df["candidate_split"].where(df["split_exclusion_reason"].eq(""), "excluded")
train = df.loc[df["final_split"].eq("train")].copy()
validation = df.loc[df["final_split"].eq("validation")].copy()
test = df.loc[df["final_split"].eq("test")].copy()
excluded = df.loc[df["final_split"].eq("excluded")].copy()

print("Training:", len(train))
print("Validation:", len(validation))
print("Testing:", len(test))
print("Set aside at Step 3:", len(excluded))
print("\nReasons:")
print(excluded["split_exclusion_reason"].value_counts().to_string())

Training: 6096
Validation: 1163
Testing: 1672
Set aside at Step 3: 739

Reasons:
split_exclusion_reason
31-day gap before validation or testing                      519
Training animal also appears in validation or test period    182
Validation animal also appears in test period                 38


## 4. Check that the split is correct

We check row counts, dates, animal overlap and label timing. These are checks of the data split, not model performance results.

Expected counts for the saved dataset and our fixed rules:

- Training: **6,096**.
- Validation: **1,163**.
- Testing: **1,672**.
- Set aside at this step: **739**.

We retain **8,931 records across the three splits**, including more than 5,000 for training. Step 2 had already set aside 270 other records. These are separate steps and must not be counted twice.

If a check fails, investigate the file or rule that changed. Do not edit the target or dates merely to make the counts match.

In [5]:
assert len(train) + len(validation) + len(test) + len(excluded) == len(df)
assert (len(train), len(validation), len(test), len(excluded)) == (6096, 1163, 1672, 739)
assert sum(map(len, [train, validation, test])) >= 5000
assert len(train) >= 5000

splits = {"train": train, "validation": validation, "test": test}
for name, part in splits.items():
    assert part["source_row_number"].is_unique, name
    assert set(part["long_stay_30"].unique()) == {0, 1}, name
    assert part["split_exclusion_reason"].eq("").all(), name

for first, second in [(train, validation), (train, test), (validation, test)]:
    assert set(first["id"]).isdisjoint(second["id"])
    assert set(first["source_row_number"]).isdisjoint(second["source_row_number"])

assert train["intake_date"].max() < TRAIN_END_EXCLUSIVE
assert validation["intake_date"].min() >= VALIDATION_START
assert validation["intake_date"].max() < VALIDATION_END_EXCLUSIVE
assert test["intake_date"].min() >= TEST_START
assert (train["label_known_date"] < VALIDATION_START).all()
assert (validation["label_known_date"] < TEST_START).all()
assert (test["label_known_date"] <= OBSERVATION_DATE).all()

print("All checks passed: no shared animals or rows between splits.")
print("Dates and label timing are consistent with the chosen periods.")
print("Retained across the three splits:", len(train) + len(validation) + len(test))

All checks passed: no shared animals or rows between splits.
Dates and label timing are consistent with the chosen periods.
Retained across the three splits: 8931


## 5. Save the files for the next notebooks

This cell creates **step3_results** beside step2_results. If you uploaded the two Step 2 files directly to Colab, it creates `/content/step3_results`.

- **train_records.csv:** use for detailed EDA and learning preprocessing rules.
- **validation_records.csv:** use later to compare and tune models.
- **test_records.csv:** reserve for the final evaluation.
- **excluded_split_records.csv:** records set aside here, with reasons.
- **split_manifest.csv:** one entry per Step 2 row showing its assigned split or exclusion reason.
- **split_notes.json:** dates, counts, fingerprints and limitations.

Rerunning this cell replaces only these generated files. In Colab, download the folder or save it to your own storage before the session ends.

**These files still contain audit and outcome columns. They are not ready-made model inputs.** Later, explicitly select the allowed arrival-time features. Do not use every column except the target.

Exclude outcome fields, final duration, derived label dates, split information and row identifiers from prediction inputs. Retain animal ID separately for future validation-fold checks. Some animal attributes may have been updated after arrival; we must review their availability too.

In [6]:
PROJECT_DIR = STEP2_DIR.parent if STEP2_DIR.name == "step2_results" else STEP2_DIR
RESULTS_DIR = PROJECT_DIR / "step3_results"
RESULTS_DIR.mkdir(exist_ok=True)

for name, part in splits.items():
    part.to_csv(RESULTS_DIR / f"{name}_records.csv", index=False)
excluded.to_csv(RESULTS_DIR / "excluded_split_records.csv", index=False)
manifest_columns = ["source_row_number", "id", "impound_number", "intake_date",
                    "label_known_date", "candidate_split", "final_split", "split_exclusion_reason"]
df[manifest_columns].to_csv(RESULTS_DIR / "split_manifest.csv", index=False)

split_notes = {
    "source_url": notes["source_url"],
    "step2_csv_sha256": source_hash,
    "original_csv_sha256": notes["source_sha256"],
    "observation_date": str(OBSERVATION_DATE.date()),
    "validation_start": str(VALIDATION_START.date()),
    "test_start": str(TEST_START.date()),
    "gap_days": GAP_DAYS,
    "training_end_exclusive": str(TRAIN_END_EXCLUSIVE.date()),
    "validation_end_exclusive": str(VALIDATION_END_EXCLUSIVE.date()),
    "rows_from_step2": len(df),
    "training_rows": len(train), "validation_rows": len(validation), "test_rows": len(test),
    "excluded_at_step3": len(excluded),
    "exclusion_reasons": {str(k): int(v) for k, v in excluded["split_exclusion_reason"].value_counts().items()},
    "animal_overlap_between_splits": 0,
    "evaluation_unit": "Admission event; the animal is absent from earlier retained splits.",
    "label_known_rule": "Short stay: outcome date. Long stay: intake plus 31 days.",
    "pandas_version_used": pd.__version__,
    "limitations": [
        "A single retrospective snapshot cannot prove historical data-entry or update times.",
        "Blank outcomes are assumed ongoing, as documented in Step 2.",
        "Removing earlier visits of later animals changes the evaluated population.",
        "Different visits of one animal may remain within one split.",
        "Feature availability, missing-value handling, encoding and scaling remain unfinished.",
        "Model tuning must use time/group-aware validation; do not replace this with random row-level cross-validation."
    ]
}
split_notes["output_sha256"] = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in [RESULTS_DIR / f"{name}_records.csv" for name in splits]
}
(RESULTS_DIR / "split_notes.json").write_text(json.dumps(split_notes, indent=2), encoding="utf-8")
assert hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest() == source_hash
print("Saved six files in:", RESULTS_DIR)
print("Step 2 input file unchanged.")

Saved six files in: C:\Users\ishin\Documents\Codex\2026-09-15\we-x20\outputs\step3_results
Step 2 input file unchanged.


## 6. What each member should explain

- **Member 1:** what training, validation and testing are for.
- **Member 2:** why we split by arrival date and leave gaps.
- **Member 3:** why replacement ages, encoding and scaling must be learned from training data only.
- **Member 4:** how the animal-overlap checks work and where the output files are saved.

Everyone should run the notebook and explain at least one example to the others. Record actual contributions and any changes your group makes.

### Simple viva explanation

"We used older records for training and later records for validation and testing. We left 31-day gaps so earlier animals had enough time to reach a known answer. We also set aside earlier records for animals appearing in a later split. This lets us evaluate future admissions from animals our model has not already seen."

### Next: Notebook 4 - Understand the training data

Use **train_records.csv** for detailed charts, relationships and preprocessing choices. We will investigate missing ages, unusual dates, categories and the target balance there. Basic structure checks across the splits are fine, but do not explore test outcomes to choose features or settings.

Later, fit imputers, encoders and scalers on training data, then apply them unchanged to validation and test data. Keep the test set reserved until model and threshold choices are fixed. If we later refit using training plus validation, the final test set remains separate.